In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert
from ipywidgets import interactive, FloatSlider, IntSlider, RadioButtons, HBox, VBox, HTML, Layout
from IPython.display import display


# ============================================================
# CAUSALITY CONSTRAINTS ON THE FREQUENCY RESPONSE
# ============================================================
#
# This notebook illustrates three important consequences of
# causality in discrete-time filters.
#
#
# PARAMETER a
# -----------
# The parameter a is the real pole location of the basic causal
# first-order system
#
#       H(z) = 1 / (1 - a z^-1)
#
# with |a| < 1.
#
# Increasing a toward 1 produces a more slowly decaying impulse
# response and a stronger low-frequency concentration.
#
#
# PARAMETER c
# -----------
# The parameter c is the pole location of the stable all-pass
# factor
#
#       A(z) = (z^-1 - c) / (1 - c z^-1).
#
# Its pole is located at z = c and its zero at z = 1/c.
#
# Since
#
#       |A(e^jw)| = 1,
#
# changing c changes the phase response without changing the
# magnitude response.
#
#
# MODE 1
# ------
# A causal impulse response can be reconstructed from its even
# component:
#
#       h[n] = 2 h_e[n] u[n] - h_e[0] delta[n].
#
#
# MODE 2
# ------
# For a real causal impulse response, the real and imaginary
# parts of H(e^{jw}) are related through the discrete periodic
# Hilbert-transform relation.
#
#
# MODE 3
# ------
# For a causal minimum-phase system, the phase response is
# determined by log |H(e^{jw})|.
#
# A stable non-minimum-phase system may have exactly the same
# magnitude response but a different phase response.
#
# In the phase-comparison graph, both phases are displayed as
# principal values in the interval [-pi, pi]. This avoids the
# additional 2*pi branches introduced by phase unwrapping and
# makes the comparison easier to interpret.
#
# ============================================================


# ============================================================
# CSS
# ============================================================

style_html = HTML("""
<style>

.causal-root {
    width: 960px;
    max-width: 960px;
    font-family: Arial, sans-serif;
}

.causal-header {
    background: #303943;
    color: white;
    padding: 8px 14px;
    border-radius: 7px 7px 0 0;
    font-size: 19px;
    font-weight: bold;
}

.causal-intro {
    background: #f5f7f8;
    border: 1px solid #d3d9dd;
    border-top: none;
    padding: 7px 12px;
    border-radius: 0 0 7px 7px;
    font-size: 12px;
    line-height: 1.50;
    margin-bottom: 6px;
}

.causal-accent {
    font-weight: bold;
    color: #3d5563;
}

.causal-controls-title {
    font-size: 12.5px;
    font-weight: bold;
    margin: 0 0 3px 3px;
    color: #303943;
}

.horizontal-radio .widget-radio-box {
    display: flex !important;
    flex-direction: row !important;
    flex-wrap: nowrap !important;
    gap: 25px !important;
}

.horizontal-radio .widget-radio-box label {
    margin: 0 !important;
    white-space: nowrap !important;
    font-weight: bold;
}

.horizontal-radio > label {
    display: none !important;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea {
    overflow-x: visible !important;
    max-width: none !important;
}

</style>
""")


# ============================================================
# VISIBLE DOCUMENTATION
# ============================================================

header_html = HTML("""
<div class="causal-root">

    <div class="causal-header">
        Causality Constraints on the Frequency Response
    </div>

    <div class="causal-intro">

        <span class="causal-accent">What happens:</span>
        causality links quantities that would otherwise appear independent.

        <span class="causal-accent">What to observe:</span>
        a causal impulse response can be reconstructed from one symmetry component,
        while the real/imaginary parts and, for minimum-phase systems,
        magnitude/phase are linked through Hilbert-transform relations.

        <br>

        <span class="causal-accent">Parameters:</span>
        <b>a</b> is the real pole location of the basic causal first-order system
        H(z) = 1/(1 - az<sup>-1</sup>);
        <b>c</b> is the pole location of the all-pass factor used in the
        magnitude-phase experiment, whose corresponding zero is located at 1/c.

        <br>

        <span class="causal-accent">Phase display:</span>
        in the magnitude-phase experiment, phase is shown using its principal value
        in the interval [-π, π], so equivalent 2π-shifted phase branches are not displayed.

    </div>

</div>
""")


# ============================================================
# AUXILIARY FUNCTIONS
# ============================================================


# ------------------------------------------------------------
# Causal exponentially decaying impulse response
#
#       h[n] = a^n u[n]
# ------------------------------------------------------------

def causal_impulse_response(a, M):

    n = np.arange(-M, M + 1)

    h = np.zeros(len(n), dtype=float)

    h[n >= 0] = a**n[n >= 0]

    return n, h


# ------------------------------------------------------------
# Even and odd components
# ------------------------------------------------------------

def even_odd_components(h):

    h_reverse = h[::-1]

    h_even = 0.5 * (h + h_reverse)

    h_odd = 0.5 * (h - h_reverse)

    return h_even, h_odd


# ------------------------------------------------------------
# Reconstruction of causal h[n] from its even component
# ------------------------------------------------------------

def reconstruct_from_even(n, h_even):

    h_rec = np.zeros(len(n), dtype=float)

    h_rec[n > 0] = 2.0 * h_even[n > 0]

    h_rec[n == 0] = h_even[n == 0]

    return h_rec


# ------------------------------------------------------------
# Periodic Hilbert reconstruction
#
#       H_I = -Hilbert{H_R}
# ------------------------------------------------------------

def recover_imaginary_from_real(H_real):

    return -np.imag(hilbert(H_real))


# ------------------------------------------------------------
# Minimum-phase first-order system
#
#       H(z) = 1 / (1 - a z^-1)
# ------------------------------------------------------------

def minimum_phase_response(a, omega):

    return 1.0 / (1.0 - a * np.exp(-1j * omega))


# ------------------------------------------------------------
# Stable first-order all-pass factor
#
#       A(z) = (z^-1 - c) / (1 - c z^-1)
#
# pole = c
# zero = 1/c
#
#       |A(e^jw)| = 1
# ------------------------------------------------------------

def allpass_response(c, omega):

    zinv = np.exp(-1j * omega)

    return (zinv - c) / (1.0 - c * zinv)


# ============================================================
# MAIN INTERACTIVE FUNCTION
# ============================================================

def plot_causality_lab(mode='Even/Odd Reconstruction', a=0.70, allpass_c=0.60, M=20):


    # ========================================================
    # MODE 1 — EVEN / ODD RECONSTRUCTION
    # ========================================================

    if mode == 'Even/Odd Reconstruction':

        n, h = causal_impulse_response(a, M)

        h_even, h_odd = even_odd_components(h)

        h_rec = reconstruct_from_even(n, h_even)


        fig, axes = plt.subplots(2, 2, figsize=(12.8, 6.9))


        ax1 = axes[0, 0]

        ax2 = axes[0, 1]

        ax3 = axes[1, 0]

        ax4 = axes[1, 1]


        # ----------------------------------------------------
        # Original causal impulse response
        # ----------------------------------------------------

        markerline, stemlines, baseline = ax1.stem(n, h, basefmt=' ')

        plt.setp(stemlines, linewidth=1.2)

        plt.setp(markerline, markersize=4.5)


        ax1.axvline(0, linestyle=':', linewidth=1.0)

        ax1.axhline(0, linewidth=0.7)

        ax1.set_title('Original Causal Impulse Response', pad=10)

        ax1.set_xlabel('Sample index n', labelpad=8)

        ax1.set_ylabel(r'$h[n]$')

        ax1.grid(True, linestyle=':', alpha=0.25)


        # ----------------------------------------------------
        # Even component
        # ----------------------------------------------------

        markerline, stemlines, baseline = ax2.stem(n, h_even, basefmt=' ')

        plt.setp(stemlines, linewidth=1.2)

        plt.setp(markerline, markersize=4.5)


        ax2.axvline(0, linestyle=':', linewidth=1.0)

        ax2.axhline(0, linewidth=0.7)

        ax2.set_title('Even Component', pad=10)

        ax2.set_xlabel('Sample index n', labelpad=8)

        ax2.set_ylabel(r'$h_e[n]$')

        ax2.grid(True, linestyle=':', alpha=0.25)


        # ----------------------------------------------------
        # Odd component
        # ----------------------------------------------------

        markerline, stemlines, baseline = ax3.stem(n, h_odd, basefmt=' ')

        plt.setp(stemlines, linewidth=1.2)

        plt.setp(markerline, markersize=4.5)


        ax3.axvline(0, linestyle=':', linewidth=1.0)

        ax3.axhline(0, linewidth=0.7)

        ax3.set_title('Odd Component', pad=10)

        ax3.set_xlabel('Sample index n', labelpad=8)

        ax3.set_ylabel(r'$h_o[n]$')

        ax3.grid(True, linestyle=':', alpha=0.25)


        # ----------------------------------------------------
        # Reconstruction from even part
        # ----------------------------------------------------

        markerline, stemlines, baseline = ax4.stem(n, h, linefmt='-', markerfmt='o', basefmt=' ')

        plt.setp(stemlines, linewidth=1.2)

        plt.setp(markerline, markersize=4.0)


        ax4.plot(n, h_rec, 'x', markersize=7, label='Reconstructed from even part')


        ax4.axvline(0, linestyle=':', linewidth=1.0)

        ax4.axhline(0, linewidth=0.7)

        ax4.set_title('Causal Reconstruction', pad=10)

        ax4.set_xlabel('Sample index n', labelpad=8)

        ax4.set_ylabel(r'$h[n]$')

        ax4.grid(True, linestyle=':', alpha=0.25)


        ax4.legend(loc='upper center', bbox_to_anchor=(0.5, -0.23), frameon=False, fontsize=10.3)


        fig.suptitle(r'$h[n]=2h_e[n]u[n]-h_e[0]\delta[n]$', fontsize=13)


        plt.subplots_adjust(left=0.07, right=0.98, top=0.88, bottom=0.13, wspace=0.25, hspace=0.66)


        plt.show()

        plt.close(fig)


    # ========================================================
    # MODE 2 — REAL / IMAGINARY PARTS
    # ========================================================

    elif mode == 'Real ↔ Imaginary Parts':

        omega = np.linspace(-np.pi, np.pi, 1024, endpoint=False)

        n = np.arange(0, 81)

        h = a**n


        H = np.exp(-1j * np.outer(omega, n)) @ h


        H_real = np.real(H)

        H_imag = np.imag(H)


        H_imag_recovered = recover_imaginary_from_real(H_real)


        error = H_imag - H_imag_recovered


        fig, axes = plt.subplots(2, 2, figsize=(12.8, 7.2))


        ax1 = axes[0, 0]

        ax2 = axes[0, 1]

        ax3 = axes[1, 0]

        ax4 = axes[1, 1]


        # ----------------------------------------------------
        # Real part
        # ----------------------------------------------------

        ax1.plot(omega, H_real, linewidth=1.7)


        ax1.set_title(r'Real Part $H_R(e^{j\omega})$', pad=10)

        ax1.set_xlabel(r'Frequency $\omega$', labelpad=8)

        ax1.set_ylabel(r'$H_R$')


        ax1.set_xlim(-np.pi, np.pi)


        ax1.set_xticks([-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi])

        ax1.set_xticklabels([r'$-\pi$', r'$-\pi/2$', '0', r'$\pi/2$', r'$\pi$'])


        ax1.grid(True, linestyle=':', alpha=0.25)


        # ----------------------------------------------------
        # Imaginary part
        # ----------------------------------------------------

        ax2.plot(omega, H_imag, linewidth=1.7, label='Direct imaginary part')

        ax2.plot(omega, H_imag_recovered, '--', linewidth=1.6, label='Recovered from real part')


        ax2.set_title(r'Imaginary Part $H_I(e^{j\omega})$', pad=10)

        ax2.set_xlabel(r'Frequency $\omega$', labelpad=8)

        ax2.set_ylabel(r'$H_I$')


        ax2.set_xlim(-np.pi, np.pi)


        ax2.set_xticks([-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi])

        ax2.set_xticklabels([r'$-\pi$', r'$-\pi/2$', '0', r'$\pi/2$', r'$\pi$'])


        ax2.grid(True, linestyle=':', alpha=0.25)


        ax2.legend(loc='upper center', bbox_to_anchor=(0.5, -0.25), ncol=2, frameon=False, fontsize=10.3, columnspacing=1.4)


        # ----------------------------------------------------
        # Direct versus reconstructed imaginary part
        # ----------------------------------------------------

        ax3.plot(omega, H_imag, linewidth=2.0, label='Direct')

        ax3.plot(omega, H_imag_recovered, '--', linewidth=1.5, label='Hilbert reconstruction')


        ax3.set_title('Direct and Reconstructed Imaginary Parts', pad=12)

        ax3.set_xlabel(r'Frequency $\omega$', labelpad=8)

        ax3.set_ylabel(r'$H_I$')


        ax3.set_xlim(-np.pi, np.pi)


        ax3.set_xticks([-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi])

        ax3.set_xticklabels([r'$-\pi$', r'$-\pi/2$', '0', r'$\pi/2$', r'$\pi$'])


        ax3.grid(True, linestyle=':', alpha=0.25)


        ax3.legend(loc='upper center', bbox_to_anchor=(0.5, -0.25), ncol=2, frameon=False, fontsize=10.3, columnspacing=1.4)


        # ----------------------------------------------------
        # Reconstruction error
        # ----------------------------------------------------

        ax4.plot(omega, error, linewidth=1.5)


        ax4.axhline(0, linestyle=':', linewidth=1.0)


        ax4.set_title('Reconstruction Error', pad=12)

        ax4.set_xlabel(r'Frequency $\omega$', labelpad=8)

        ax4.set_ylabel('Error')


        ax4.set_xlim(-np.pi, np.pi)


        ax4.set_xticks([-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi])

        ax4.set_xticklabels([r'$-\pi$', r'$-\pi/2$', '0', r'$\pi/2$', r'$\pi$'])


        ax4.grid(True, linestyle=':', alpha=0.25)


        fig.suptitle(r'Causality: $H_I(e^{j\omega})=-\mathcal{H}\{H_R(e^{j\omega})\}$', fontsize=13)


        plt.subplots_adjust(left=0.07, right=0.98, top=0.88, bottom=0.14, wspace=0.25, hspace=0.78)


        plt.show()

        plt.close(fig)


    # ========================================================
    # MODE 3 — MAGNITUDE / PHASE AND MINIMUM PHASE
    # ========================================================

    elif mode == 'Magnitude ↔ Phase':

        omega = np.linspace(-np.pi, np.pi, 2048, endpoint=False)


        # ----------------------------------------------------
        # Minimum-phase system
        # ----------------------------------------------------

        H_min = minimum_phase_response(a, omega)


        # ----------------------------------------------------
        # Stable all-pass modification
        # ----------------------------------------------------

        A_allpass = allpass_response(allpass_c, omega)


        H_nonmin = H_min * A_allpass


        magnitude_min = np.abs(H_min)

        magnitude_nonmin = np.abs(H_nonmin)


        # ----------------------------------------------------
        # Principal phase values
        #
        # np.angle returns phase in [-pi, pi].
        #
        # We deliberately do NOT use np.unwrap here because the
        # purpose of this graph is to compare the principal phase
        # responses directly without displaying equivalent 2*pi
        # branches.
        # ----------------------------------------------------

        phase_min = np.angle(H_min)

        phase_nonmin = np.angle(H_nonmin)


        # ----------------------------------------------------
        # Recover minimum-phase phase from log magnitude
        # ----------------------------------------------------

        log_magnitude = np.log(np.maximum(magnitude_min, 1e-14))


        phase_recovered = -np.imag(hilbert(log_magnitude))


        # Map the reconstructed phase to its principal interval
        # for direct comparison with np.angle(H_min).

        phase_recovered = np.angle(np.exp(1j * phase_recovered))


        fig, axes = plt.subplots(2, 2, figsize=(12.8, 7.3))


        ax1 = axes[0, 0]

        ax2 = axes[0, 1]

        ax3 = axes[1, 0]

        ax4 = axes[1, 1]


        # ----------------------------------------------------
        # Magnitude responses
        # ----------------------------------------------------

        ax1.plot(omega, magnitude_min, linewidth=2.0, label='Minimum phase')

        ax1.plot(omega, magnitude_nonmin, '--', linewidth=1.5, label='Non-minimum phase')


        ax1.set_title('Magnitude Responses', pad=10)

        ax1.set_xlabel(r'Frequency $\omega$', labelpad=8)

        ax1.set_ylabel(r'$|H(e^{j\omega})|$')


        ax1.set_xlim(-np.pi, np.pi)


        ax1.set_xticks([-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi])

        ax1.set_xticklabels([r'$-\pi$', r'$-\pi/2$', '0', r'$\pi/2$', r'$\pi$'])


        ax1.grid(True, linestyle=':', alpha=0.25)


        ax1.legend(loc='upper center', bbox_to_anchor=(0.5, -0.25), ncol=2, frameon=False, fontsize=10.4, columnspacing=1.5)


        # ----------------------------------------------------
        # Principal phase responses
        # ----------------------------------------------------

        ax2.plot(omega, phase_min, linewidth=1.8, label='Minimum phase')

        ax2.plot(omega, phase_nonmin, '--', linewidth=1.6, label='Same magnitude + all-pass')


        ax2.set_title('Principal Phase Responses', pad=10)

        ax2.set_xlabel(r'Frequency $\omega$', labelpad=8)

        ax2.set_ylabel('Phase [rad]')


        ax2.set_xlim(-np.pi, np.pi)

        ax2.set_ylim(-np.pi, np.pi)


        ax2.set_xticks([-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi])

        ax2.set_xticklabels([r'$-\pi$', r'$-\pi/2$', '0', r'$\pi/2$', r'$\pi$'])


        ax2.set_yticks([-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi])

        ax2.set_yticklabels([r'$-\pi$', r'$-\pi/2$', '0', r'$\pi/2$', r'$\pi$'])


        ax2.grid(True, linestyle=':', alpha=0.25)


        ax2.legend(loc='upper center', bbox_to_anchor=(0.5, -0.25), ncol=2, frameon=False, fontsize=10.4, columnspacing=1.5)


        # ----------------------------------------------------
        # Minimum-phase reconstruction
        # ----------------------------------------------------

        ax3.plot(omega, phase_min, linewidth=2.0, label='Actual minimum-phase phase')

        ax3.plot(omega, phase_recovered, '--', linewidth=1.6, label='Recovered from log magnitude')


        ax3.set_title('Minimum-Phase Reconstruction', pad=12)

        ax3.set_xlabel(r'Frequency $\omega$', labelpad=8)

        ax3.set_ylabel('Phase [rad]')


        ax3.set_xlim(-np.pi, np.pi)

        ax3.set_ylim(-np.pi, np.pi)


        ax3.set_xticks([-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi])

        ax3.set_xticklabels([r'$-\pi$', r'$-\pi/2$', '0', r'$\pi/2$', r'$\pi$'])


        ax3.set_yticks([-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi])

        ax3.set_yticklabels([r'$-\pi$', r'$-\pi/2$', '0', r'$\pi/2$', r'$\pi$'])


        ax3.grid(True, linestyle=':', alpha=0.25)


        ax3.legend(loc='upper center', bbox_to_anchor=(0.5, -0.25), ncol=2, frameon=False, fontsize=10.4, columnspacing=1.4)


        # ----------------------------------------------------
        # Interpretation panel
        # ----------------------------------------------------

        ax4.axis('off')


        zero_allpass = 1.0 / allpass_c


        text_left = (
            'MINIMUM-PHASE SYSTEM\n'
            '──────────────────\n'
            f'Pole a       : {a:.3f}\n'
            'Location     : inside unit circle\n'
            'Zeros        : none\n\n'
            'The phase is determined\n'
            'by log |H| through the\n'
            'Hilbert relation.'
        )


        text_right = (
            'ALL-PASS MODIFICATION\n'
            '──────────────────\n'
            f'Pole c       : {allpass_c:.3f}\n'
            f'Zero 1/c     : {zero_allpass:.3f}\n'
            'Zero         : outside unit circle\n\n'
            'Magnitude    : unchanged\n'
            'Phase        : changed\n\n'
            'Magnitude alone does not\n'
            'determine an arbitrary\n'
            'causal phase.'
        )


        ax4.text(0.00, 0.96, text_left, transform=ax4.transAxes, ha='left', va='top', fontsize=9.2, family='monospace', linespacing=1.45)


        ax4.text(0.62, 0.96, text_right, transform=ax4.transAxes, ha='left', va='top', fontsize=9.2, family='monospace', linespacing=1.45)


        fig.suptitle('Magnitude–Phase Relation and the Minimum-Phase Condition', fontsize=13)


        plt.subplots_adjust(left=0.07, right=0.98, top=0.88, bottom=0.14, wspace=0.28, hspace=0.80)


        plt.show()

        plt.close(fig)


# ============================================================
# SLIDER APPEARANCE
# ============================================================

slider_style = {'description_width': '85px'}


# ============================================================
# RADIO BUTTONS
# ============================================================

mode_selector = RadioButtons(
    options=[
        'Even/Odd Reconstruction',
        'Real ↔ Imaginary Parts',
        'Magnitude ↔ Phase'
    ],
    value='Even/Odd Reconstruction',
    description='',
    layout=Layout(width='720px')
)


mode_selector.add_class('horizontal-radio')


# ============================================================
# SLIDERS
#
# continuous_update=True causes interactive(...) to call the
# plotting function continuously while the slider is moving.
# ============================================================

a_slider = FloatSlider(
    value=0.70,
    min=0.10,
    max=0.95,
    step=0.05,
    description='Pole a:',
    continuous_update=True,
    readout_format='.2f',
    style=slider_style,
    layout=Layout(width='310px')
)


allpass_slider = FloatSlider(
    value=0.60,
    min=0.10,
    max=0.90,
    step=0.05,
    description='All-pass c:',
    continuous_update=True,
    readout_format='.2f',
    style=slider_style,
    layout=Layout(width='330px')
)


M_slider = IntSlider(
    value=20,
    min=8,
    max=35,
    step=1,
    description='Samples:',
    continuous_update=True,
    style=slider_style,
    layout=Layout(width='300px')
)


# ============================================================
# ENABLE / DISABLE CONTROLS ACCORDING TO MODE
# ============================================================

def update_control_state(change=None):

    current_mode = mode_selector.value


    if current_mode == 'Even/Odd Reconstruction':

        a_slider.disabled = False

        allpass_slider.disabled = True

        M_slider.disabled = False


    elif current_mode == 'Real ↔ Imaginary Parts':

        a_slider.disabled = False

        allpass_slider.disabled = True

        M_slider.disabled = True


    elif current_mode == 'Magnitude ↔ Phase':

        a_slider.disabled = False

        allpass_slider.disabled = False

        M_slider.disabled = True


mode_selector.observe(update_control_state, names='value')


update_control_state()


# ============================================================
# INTERACTIVE OBJECT
# ============================================================

widget_plot = interactive(
    plot_causality_lab,
    mode=mode_selector,
    a=a_slider,
    allpass_c=allpass_slider,
    M=M_slider
)


plot_output = widget_plot.children[-1]


plot_output.layout = Layout(
    width='auto',
    overflow='visible'
)


# ============================================================
# CONTROL LAYOUT
# ============================================================

mode_box = VBox(
    [
        HTML("<div class='causal-controls-title'>Select experiment</div>"),
        mode_selector
    ],
    layout=Layout(
        width='950px',
        border='1px solid #d3d9dd',
        padding='5px 8px',
        overflow='visible'
    )
)


parameter_row = HBox(
    [
        a_slider,
        allpass_slider,
        M_slider
    ],
    layout=Layout(
        width='950px',
        justify_content='space-between',
        align_items='center'
    )
)


parameter_box = VBox(
    [
        HTML("<div class='causal-controls-title'>Experiment parameters</div>"),
        parameter_row
    ],
    layout=Layout(
        width='950px',
        border='1px solid #d3d9dd',
        padding='5px 8px',
        overflow='visible'
    )
)


# ============================================================
# FINAL LAYOUT
# ============================================================

main_layout = VBox(
    [
        header_html,
        mode_box,
        parameter_box,
        plot_output
    ],
    layout=Layout(
        width='960px',
        overflow='visible',
        align_items='flex-start'
    )
)


# ============================================================
# DISPLAY
# ============================================================

display(style_html)

display(main_layout)